# 03 — Robust Retrieval Debugging (legacy union-gate)

Compare direct FAISS search with typo-aware hybrid retrieval and inspect abstentions. Run from the repository root. The actual embedding model is required for the live cells.

In [ ]:
from src.rag.query_normalizer import normalize_query
from src.rag.retriever import RobustRetriever
from src.rag.answer_service import answer_question
from src.rag.pipeline import build_pipeline


## 1. Query normalization is independent of FAISS

In [ ]:
queries = [
    "Who is Athena?",
    "how are apolo and atemis related?",
    "Which Greek deity represents clever communication and traveling?",
    "What smartphone did Zeus use?",
]
for q in queries:
    normalized, corrections = normalize_query(q)
    print(q, "=>", normalized, "| edits:", corrections)


## 2. Load your local embedding model and FAISS index

First use may download the embedding model. Build the index with `python -m src.rag.build_index` if needed.

In [ ]:
store = build_pipeline()
retriever = RobustRetriever(store)
print("Indexed passages:", len(store.chunks))


## 3. Compare the old and new rankings on a typo

In [ ]:
question = "how are apolo and atemis related?"
print("raw FAISS baseline BASELINE:")
for hit in store.search(question, top_k=4, min_score=0.25):
    print(round(hit.score, 3), hit.chunk.document_id)
print("\nlegacy union-gate ROBUST RETRIEVAL:")
result = retriever.retrieve(question)
print(result.normalized_question, result.status, result.reason)
for hit in result.hits:
    print(round(hit.score, 3), hit.chunk.document_id)


## 4. The evidence gate: abstention is not the same as no search results

In [ ]:
for question in queries:
    result = answer_question(question, store)
    print("QUESTION:", question)
    print("MODE:", result.mode, "REASON:", result.reason)
    print("EVIDENCE:", [h.chunk.document_id for h in result.hits])
    print()


## 5. Benchmark on corpus-derived regression questions

These are small regression checks, NOT general accuracy estimates.

In [ ]:
import json
from pathlib import Path
from src.rag.evaluate import evaluate_retrieval, evaluate_robust_retrieval
payload = json.loads(Path("data/evaluation/retrieval_regression.json").read_text(encoding="utf-8"))
print("raw FAISS baseline:", evaluate_retrieval(store, payload["positive"], k=5))
print("legacy union-gate:", evaluate_robust_retrieval(store, payload["positive"], payload["negative"], k=5))


## Exercise

Add five newly phrased questions that you did NOT use when designing the rules. Report false rejects and false accepts, not just Hit@5. A high retrieval cosine score is not a confidence of answer correctness.